# Germany vs Belgium vs Netherlands vs Switzerland: whose trains are late?

Every railway grades itself with its own rules. Switzerland counts a train as on time if it's
less than 3 minutes late, Germany and Belgium under 6, Romania under 20. Most only check the last
stop, and cancelled trains usually don't count at all. So this notebook measures four countries
with **one ruler**:

* **every arrival at every station** inside the country, not just the last stop
* **on time = less than 6 minutes late** (Deutsche Bahn's own definition, 5:59)
* **cancelled stops are counted**: separately, and once as a fail

Period: **1 November 2025 to 31 October 2026** (Deutsche Bahn's full-network data starts in
November 2025). About 242 million arrivals, about 26 GB of downloads.

The notebook has three parts:
1. **Download** the public data (skip it if you already have the files)
2. **Prepare** it into one standard table per country and month (`data/std/*.parquet`)
3. **Analyse**: every number in the video

In [ ]:
import glob
import os
import subprocess
import tempfile
import zipfile
from pathlib import Path

import duckdb
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import requests

DATA = Path("data")
STD = DATA / "std"
MONTHS = [f"{y}-{m:02d}" for y, m in [(2025, 11), (2025, 12)] + [(2026, k) for k in range(1, 11)]]
NAMES = {"DE": "Germany", "BE": "Belgium", "NL": "Netherlands", "CH": "Switzerland"}
ORDER = ["DE", "BE", "NL", "CH"]

plt.rcParams.update({"figure.figsize": (10, 4.5), "axes.spines.top": False,
                     "axes.spines.right": False, "axes.grid": True, "grid.alpha": 0.3})
BLUE, ORANGE = "#4F79B8", "#E8641D"
pd.set_option("display.float_format", "{:.4f}".format)

## 1. Download

| Country | Source | Size |
|---|---|---|
| Germany | Deutsche Bahn timetable API, collected every few minutes by [piebro/deutsche-bahn-data](https://huggingface.co/datasets/piebro/deutsche-bahn-data) (CC BY 4.0) | 4.3 GB |
| Netherlands | [Rijden de Treinen](https://www.rijdendetreinen.nl/en/open-data) service archive (CC BY 4.0) | 0.4 GB |
| Switzerland | [opentransportdata.swiss](https://archive.opentransportdata.swiss) actual data (Ist-Daten) archive. Includes buses and trams, hence the size | 17 GB |
| Belgium | [Infrabel open data](https://opendata.infrabel.be): raw punctuality files, plus the official punctuality and cancellation series | 4 GB |

Files that are already there are skipped, so you can stop and restart it.

In [ ]:
SOURCES = {
    "nl": ("services-{m}.csv.gz", "https://opendata.rijdendetreinen.nl/public/services/services-{m}.csv.gz"),
    "de": ("data-{m}.parquet", "https://huggingface.co/datasets/piebro/deutsche-bahn-data/resolve/main/"
                               "monthly_processed_data/data-{m}.parquet"),
    "ch": ("ist-daten-v2-{m}.zip", "https://archive.opentransportdata.swiss/istdaten/{y}/ist-daten-v2-{m}.zip"),
    "be": ("raw-punctuality-{m}.csv", "https://fr.ftp.opendatasoft.com/infrabel/PunctualityHistory/"
                                      "Data_raw_punctuality_{ym}.csv"),
}
EXTRA = {   # small files: the Dutch station list and Infrabel's own official series
    "nl/stations-nl.csv": "https://opendata.rijdendetreinen.nl/public/stations/stations-2023-09.csv",
    "be/official-punctuality.csv": "https://opendata.infrabel.be/api/explore/v2.1/catalog/datasets/"
                                   "nationale-stiptheid-per-maand/exports/csv?delimiter=%3B",
    "be/official-cancellations.csv": "https://opendata.infrabel.be/api/explore/v2.1/catalog/datasets/"
                                     "afgeschafte-treinen-per-maand-vanaf-2020/exports/csv?delimiter=%3B",
}


def fetch(url, path):
    path.parent.mkdir(parents=True, exist_ok=True)
    if path.exists() and path.stat().st_size > 0:
        return
    tmp = path.with_suffix(path.suffix + ".part")
    with requests.get(url, stream=True, timeout=120) as r:
        r.raise_for_status()
        with open(tmp, "wb") as f:
            for chunk in r.iter_content(1 << 20):
                f.write(chunk)
    tmp.rename(path)
    print(f"  {path}  ({path.stat().st_size / 1e6:,.0f} MB)")


def download():
    for rel, url in EXTRA.items():
        fetch(url, DATA / rel)
    for country, (name, url) in SOURCES.items():
        for m in MONTHS:
            fetch(url.format(m=m, y=m[:4], ym=m.replace("-", "")), DATA / country / name.format(m=m))
        print(f"{country} done")


download()

## 2. Prepare: one standard table per country and month

Every row is one train **arriving** at one station inside that country:

`country, day, hour, weekday, category, operator, train_type, station, run_id, seq, delay_min, cancelled, cut_after`

The same rules for every country:
* arrivals only (the first stop of a run has no arrival), stations inside the country only
* trains only: rail-replacement buses, extra and charter trains are dropped
* `delay_min` = actual (or last known) arrival minus planned arrival, in whole minutes rounded
  down; early arrivals count as 0
* `category`: `long` = long-distance trains (ICE, IC, EC, Nightjet, international...),
  `regional` = everything else
* `cut_after` = the train arrived here, was scheduled to go further, and never served another
  stop inside the country: it was **terminated early** at this station

Country quirks:
* **Germany**: `-9999` placeholder delays are dropped; station names lost their umlauts in the
  source.
* **Netherlands**: a few delays carry a +1440 (24 h) wrap-around around midnight.
* **Switzerland**: only arrivals marked as measured (`REAL`) or estimated (`GESCHAETZT`) are used.
  A few daily files inside the monthly zips are corrupt: they're skipped and listed.
* **Belgium**: Infrabel's raw files only contain trains that ran, so **fully cancelled trains are
  missing**. Skipped planned stops count as cancelled arrivals. An early termination shows up as
  a last row whose plan said "stop and continue" (`THOP1_COD '='`) with no actual operation.

This takes a while (about an hour, mostly the Swiss files). Months that are already prepared are
skipped.

In [ ]:
STD.mkdir(parents=True, exist_ok=True)


def sql_list(xs):
    return "(" + ",".join("'" + x.replace("'", "''") + "'" for x in xs) + ")"


LONG_DE = ("ICE", "IC", "EC", "ECE", "RJ", "RJX", "NJ", "EN", "FLX", "TGV", "EST", "ES", "WB")
LONG_NL = ("Intercity", "Intercity direct", "ICE International", "Eurostar", "Nightjet",
           "European Sleeper", "Thalys", "EuroCity", "EuroCity Direct", "Eurocity Direct",
           "ICE", "Nachttrein", "Int. Trein")
NOT_TRAIN_NL = ("Stopbus ipv trein", "Snelbus ipv trein", "Taxibus ipv trein", "Metro ipv trein",
                "Toegankelijke bus", "Drempelvrije bus", "Bus", "Speciale Trein")
LONG_CH = ("IC", "ICE", "EC", "IR", "IRE", "TGV", "RJX", "RJ", "EN", "NJ", "PE", "EXT")
LONG_BE = ("IC", "EURST", "INT", "ICE", "TGV")
NOT_TRAIN_BE = ("EXTRA", "CHARTER")

# cut_after for DE / NL / CH: the train arrived, more stops were planned, and none of them ran
CUT_AFTER = '''
  coalesce(not cancelled
    and count(*) over w_later > 0
    and not coalesce(bool_or(not cancelled) over w_later, false), false) as cut_after'''
W_LATER = "window w_later as (partition by run_id order by seq rows between 1 following and unbounded following)"
COLS = ("country, day, hour, weekday, category, operator, train_type, station, run_id, seq, "
        "delay_min, cancelled")


def p(path):
    return str(path).replace("\\", "/")

In [ ]:
def prep_de(month, out):
    src = p(DATA / "de" / f"data-{month}.parquet")
    duckdb.sql(f'''
    copy (
      with a as (
        select *, date_diff('minute', arrival_planned_time, arrival_change_time) as d
        from read_parquet('{src}')
        where arrival_planned_time is not null
          and substr(eva, 1, 3) = '080'                       -- stations in Germany
          and coalesce(train_type, '') <> 'Bus'
          and strftime(arrival_planned_time, '%Y-%m') = '{month}'
          and not coalesce(is_additional_stop, false)
      ),
      b as (
        select 'DE' as country, cast(arrival_planned_time as date) as day,
          hour(arrival_planned_time) as hour, isodow(arrival_planned_time) as weekday,
          case when train_type in {sql_list(LONG_DE)} then 'long' else 'regional' end as category,
          null::varchar as operator, train_type, station_name as station,
          regexp_replace(id, '-[0-9]+$', '') as run_id,     -- <ride id>-<run start>; drop the stop no
          train_line_station_num as seq,
          case when arrival_is_canceled then null else greatest(d, 0) end as delay_min,
          arrival_is_canceled as cancelled
        from a
        where arrival_is_canceled or (d is not null and d > -60)   -- drops the -9999 placeholder
      )
      select {COLS}, {CUT_AFTER} from b {W_LATER}
    ) to '{p(out)}' (format parquet)''')


def prep_nl(month, out):
    src, stations = p(DATA / "nl" / f"services-{month}.csv.gz"), p(DATA / "nl" / "stations-nl.csv")
    duckdb.sql(f'''
    copy (
      with st as (select code from read_csv('{stations}', header=true) where country = 'NL'),
      b as (
        select cast("Service:Date" as date) as day,
          cast("Stop:Arrival time" as timestamptz) as t_arr,
          "Service:Type" as train_type, "Service:Company" as operator,
          "Stop:Station name" as station, "Stop:Station code" as code,
          "Service:RDT-ID" as run_id, cast("Stop:RDT-ID" as bigint) as seq,
          try_cast("Stop:Arrival delay" as integer) as d,
          coalesce("Stop:Arrival cancelled", 'false') = 'true' as cancelled
        from read_csv('{src}', header=true, all_varchar=true)
        where "Stop:Arrival time" is not null and "Stop:Arrival time" <> ''
      ),
      c as (
        select 'NL' as country, day, hour(timezone('Europe/Amsterdam', t_arr)) as hour,
          isodow(day) as weekday,
          case when train_type in {sql_list(LONG_NL)} then 'long' else 'regional' end as category,
          operator, train_type, station, run_id, seq,
          case when cancelled then null
               else greatest(case when d >= 1440 then d - 1440 else d end, 0) end as delay_min,
          cancelled
        from b
        where code in (select code from st)
          and train_type not in {sql_list(NOT_TRAIN_NL)}
          and strftime(day, '%Y-%m') = '{month}'
          and (cancelled or d is not null)
      )
      select {COLS}, {CUT_AFTER} from c {W_LATER}
    ) to '{p(out)}' (format parquet)''')


def prep_ch(month, out):
    src = DATA / "ch" / f"ist-daten-v2-{month}.zip"
    tmpdir = Path(tempfile.mkdtemp(prefix="ch_", dir=DATA / "ch"))
    parts = []
    with zipfile.ZipFile(src) as z:
        for i, name in enumerate(sorted(n for n in z.namelist() if n.lower().endswith(".csv"))):
            try:
                path = Path(z.extract(name, tmpdir))
            except zipfile.BadZipFile:
                print(f"  skipped corrupt day {name}: re-download the month to include it")
                continue
            head = path.read_bytes()[:5_000_000]
            try:
                head.decode("utf-8"); enc = "utf-8"
            except UnicodeDecodeError as e:
                enc = "utf-8" if e.start > len(head) - 4 else "latin-1"
            part = tmpdir / f"part-{i:02d}.parquet"
            duckdb.sql(f'''
            copy (
              with a as (
                select * from read_csv('{p(path)}', delim=';', header=true, all_varchar=true,
                                       encoding='{enc}', quote='"')
                where PRODUKT_ID = 'Zug' and substr(BPUIC, 1, 2) = '85'      -- trains, in Switzerland
                  and coalesce(ANKUNFTSZEIT, '') <> ''
                  and coalesce(ZUSATZFAHRT_TF, 'false') = 'false'
                  and coalesce(DURCHFAHRT_TF, 'false') = 'false'
              ),
              b as (
                select strptime(BETRIEBSTAG, '%d.%m.%Y')::date as day,
                  strptime(ANKUNFTSZEIT, '%d.%m.%Y %H:%M') as t_plan,
                  try_strptime(AN_PROGNOSE, ['%d.%m.%Y %H:%M:%S', '%d.%m.%Y %H:%M']) as t_act,
                  AN_PROGNOSE_STATUS as status,
                  coalesce(VERKEHRSMITTEL_TEXT, LINIEN_TEXT) as train_type,
                  BETREIBER_ABK as operator, HALTESTELLEN_NAME as station,
                  BETRIEBSTAG || '|' || FAHRT_BEZEICHNER as run_id,
                  FAELLT_AUS_TF = 'true' as cancelled
                from a
              ),
              c as (
                select 'CH' as country, day, hour(t_plan) as hour, isodow(day) as weekday,
                  case when train_type in {sql_list(LONG_CH)} then 'long' else 'regional' end as category,
                  operator, train_type, station, run_id, epoch(t_plan)::bigint as seq,
                  case when cancelled then null
                       else greatest(floor(date_diff('second', t_plan, t_act) / 60.0), 0)::integer
                  end as delay_min,
                  cancelled
                from b
                -- measured or estimated arrivals only; forecasts would flatter the numbers
                where cancelled or (t_act is not null and status in ('REAL', 'GESCHAETZT'))
              )
              select {COLS}, {CUT_AFTER} from c {W_LATER}
            ) to '{p(part)}' (format parquet)''')
            path.unlink()
            parts.append(p(part))
    duckdb.sql(f"copy (select * from read_parquet({parts!r})) to '{p(out)}' (format parquet)")
    for f in tmpdir.rglob("*"):
        if f.is_file():
            f.unlink()
    for d in sorted(tmpdir.rglob("*"), reverse=True):
        d.rmdir()
    tmpdir.rmdir()


def prep_be(month, out):
    src = p(DATA / "be" / f"raw-punctuality-{month}.csv")
    duckdb.sql(f'''
    copy (
      with a as (
        select *, strptime(DATDEP, '%d%b%Y')::date as day,
          try_strptime(PLANNED_DATE_ARR || ' ' || PLANNED_TIME_ARR, '%d%b%Y %H:%M:%S') as t_arr,
          try_strptime(PLANNED_DATE_DEP || ' ' || PLANNED_TIME_DEP, '%d%b%Y %H:%M:%S') as t_dep,
          try_cast(DELAY_ARR as integer) as d_sec, split_part(RELATION, ' ', 1) as rel
        from read_csv('{src}', header=true, all_varchar=true)
      ),
      b as (
        select *, DATDEP || '|' || TRAIN_NO as run_id,
          row_number() over (partition by DATDEP, TRAIN_NO order by coalesce(t_arr, t_dep)) as seq,
          count(*) over (partition by DATDEP, TRAIN_NO) as n
        from a where rel not in {sql_list(NOT_TRAIN_BE)}
      ),
      c as (
        select *,
          coalesce(THOP1_COD = '=' and OP1_COD in ('P', 'D'), false) as skipped,     -- planned stop passed
          coalesce(seq = n and seq > 1 and OP1_COD is null and THOP1_COD = '=', false) as cut_here
        from b
        where t_arr is not null
          and (OP1_COD = '=' or (seq = n and OP1_COD is null) or (THOP1_COD = '=' and OP1_COD in ('P', 'D')))
      )
      select 'BE' as country, day, hour(t_arr) as hour, isodow(day) as weekday,
        case when rel in {sql_list(LONG_BE)} then 'long' else 'regional' end as category,
        TRAIN_SERV as operator, case when rel like 'S%' then 'S' else rel end as train_type,
        PTCAR_LG_NM_NL as station, run_id, seq,
        case when skipped then null else greatest(floor(d_sec / 60.0), 0)::integer end as delay_min,
        skipped as cancelled, cut_here as cut_after
      from c
      where strftime(day, '%Y-%m') = '{month}' and (skipped or d_sec is not null)
    ) to '{p(out)}' (format parquet)''')


PREP = {"de": prep_de, "nl": prep_nl, "ch": prep_ch, "be": prep_be}
for country, fn in PREP.items():
    for m in MONTHS:
        out = STD / f"{country}-{m}.parquet"
        if not out.exists():
            fn(m, out)
            n = duckdb.sql(f"select count(*) from '{p(out)}'").fetchall()[0][0]
            print(f"{country} {m}: {n:,} arrivals")
print("prepared:", len(list(STD.glob("*.parquet"))), "of", 4 * len(MONTHS), "files")

## 3. Analyse

All 48 tables are read as one view. `MEASURES` is the ruler, the same for every country:

* `under6` = share of arrivals that ran and were less than 6 minutes late (0–5 whole minutes)
* `under3` = the same at 3 minutes (the Swiss and new Dutch definition)
* `late15`, `late30` = 15+ and 30+ minutes late
* `cancelled` = share of arrivals cancelled; `under6_incl_cancel` counts a cancelled stop as a fail

In [ ]:
con = duckdb.connect()
con.sql(f"create view a as select * from read_parquet('{p(STD)}/*-*.parquet', union_by_name=true)")

MEASURES = '''
  count(*) as arrivals,
  avg(cancelled::int) as cancelled,
  avg((delay_min <= 2)::int) filter (where not cancelled) as under3,
  avg((delay_min <= 5)::int) filter (where not cancelled) as under6,
  avg((delay_min >= 15)::int) filter (where not cancelled) as late15,
  avg((delay_min >= 30)::int) filter (where not cancelled) as late30,
  avg(delay_min) filter (where not cancelled) as mean_delay,
  avg((not cancelled and delay_min <= 2)::int) as under3_incl_cancel,
  avg((not cancelled and delay_min <= 5)::int) as under6_incl_cancel'''


def q(sql):
    return con.sql(sql).df()


def hbar(series, title, fmt="{:.0%}", hot=None, xmax=1.0):
    labels, values = list(series.index)[::-1], [float(v) for v in series.values][::-1]
    fig, ax = plt.subplots(figsize=(10, 0.6 * len(values) + 1.2))
    ax.barh(labels, values, color=[ORANGE if k == hot else BLUE for k in labels])
    ax.set(title=title, xlim=(0, xmax))
    for i, v in enumerate(values):
        ax.text(v + xmax * 0.01, i, fmt.format(v), va="center")
    plt.show()


cov = q('''select country, min(day) first_day, max(day) last_day, count(distinct day) n_days,
                  count(*) arrivals, count(distinct station) stations from a group by 1''').set_index("country")
print(f"{cov.arrivals.sum():,} arrivals")
cov.loc[ORDER]

## Finding 1: the same ruler

In [ ]:
allc = q(f"select country, {MEASURES} from a group by 1").set_index("country")
cat = q(f"select country, category, {MEASURES} from a group by 1, 2").set_index(["country", "category"])

hbar(allc.under6.loc[["CH", "NL", "BE", "DE"]].rename(NAMES),
     "All trains, every stop: arrivals less than 6 minutes late", hot="Germany")
long = cat.xs("long", level="category").under6.loc[["CH", "NL", "BE", "DE"]]
hbar(long.rename(NAMES), "Long-distance trains only (ICE, IC, EC, international)", hot="Germany")
allc.loc[ORDER, ["arrivals", "under6", "under3", "late15", "cancelled"]]

## Finding 2: Belgium, three numbers for one year

Infrabel's official figure checks the final stop (and Brussels for trains that cross the
North–South junction), leaves cancelled trains out, and is also published "neutralised", with
delays blamed on outside causes removed. Infrabel says the neutralised figure isn't meant to
show what passengers experience.

In [ ]:
be_p = pd.read_csv(DATA / "be" / "official-punctuality.csv", sep=";", encoding="utf-8-sig")
be_p = be_p[be_p.maand.between(MONTHS[0], MONTHS[-1])]
belgium = pd.Series({"My measure (every stop)": allc.loc["BE", "under6"],
                     "Infrabel, official": be_p.reg.sum() / be_p.tel.sum(),
                     "Infrabel, neutralised": be_p.reg_neutr.sum() / be_p.tel.sum()})
hbar(belgium, "Belgian trains less than 6 minutes late, Nov 2025 – Oct 2026", fmt="{:.1%}")

## Finding 3: it's the ICE

In [ ]:
dist = q('''select country, category,
       avg((delay_min <= 5)::int) filter (where not cancelled) under6,
       avg((delay_min between 6 and 14)::int) filter (where not cancelled) b6_14,
       avg((delay_min >= 15)::int) filter (where not cancelled) late15
     from a group by 1, 2''').set_index(["country", "category"])
mix = (100 * dist.xs("long", level="category").loc[ORDER]).round().astype(int).rename(NAMES)
print("Out of 100 long-distance arrivals:")
display(mix)

rl = cat.under6.unstack().loc[ORDER].rename(NAMES)[["regional", "long"]]
ax = rl.plot.bar(color=[BLUE, ORANGE], rot=0, title="Regional vs long-distance: arrivals under 6 min late")
ax.set_ylim(0, 1)
ax.yaxis.set_major_formatter("{x:.0%}")
plt.show()
de = cat.loc["DE"]
print(f"Germany: regional {de.loc['regional', 'under6']:.0%}, long-distance {de.loc['long', 'under6']:.0%}, "
      f"long-distance 30+ min late {de.loc['long', 'late30']:.0%}")

## Credibility check: my numbers vs Deutsche Bahn's own

DB publishes its long-distance punctuality every month (every stop, under 6 minutes), on
[deutschebahn.com, Pünktlichkeitswerte](https://www.deutschebahn.com/de/konzern/konzernprofil/zahlen_fakten/puenktlichkeitswerte-6878476).
The values below were typed in from that page (October 2026 was provisional).

In [ ]:
DB_OFFICIAL = {"2025-11": 0.545, "2025-12": 0.617, "2026-01": 0.521, "2026-02": 0.594,
               "2026-03": 0.621, "2026-04": 0.644, "2026-05": 0.611, "2026-06": 0.526,
               "2026-07": 0.526, "2026-08": 0.542, "2026-09": 0.524, "2026-10": 0.562}
mine = q(f'''select strftime(day, '%Y-%m') ym, {MEASURES} from a
             where country = 'DE' and category = 'long' group by 1 order by 1''').set_index("ym").under6
check = pd.DataFrame({"DB official": pd.Series(DB_OFFICIAL), "My measure": mine})
check["gap"] = check["DB official"] - check["My measure"]

ax = check[["DB official", "My measure"]].plot(color=[BLUE, ORANGE], lw=3, marker="o",
                                               title="German long-distance trains under 6 min late, per month")
ax.yaxis.set_major_formatter("{x:.0%}")
plt.show()
print(f"Gap: {check.gap.min():.1%} to {check.gap.max():.1%}, correlation {check['DB official'].corr(check['My measure']):.3f}")
check

## Finding 4: where and when

In [ ]:
big = q(f'''with s as (select country, station, {MEASURES} from a group by 1, 2)
            select * from (select *, row_number() over (partition by country order by arrivals desc) r from s)
            where r <= 40''')
de_st = big[big.country == "DE"].sort_values("under6").set_index("station").under6
print("Germany's 40 busiest stations, worst five and best three (station names lost their umlauts in the source):")
display(pd.concat([de_st.head(5), de_st.tail(3)]))

hour = q(f"select country, hour, {MEASURES} from a group by 1, 2").pivot(index="hour", columns="country",
                                                                        values="under6")
order = [(h + 4) % 24 for h in range(24)]                 # the day from 04:00
ax = hour.loc[order, ORDER].rename(columns=NAMES).reset_index(drop=True).plot(
    color=[ORANGE, BLUE, "#7FA3D6", "#2E4F80"], lw=2.5, title="Arrivals under 6 min late, by hour (from 04:00)")
ax.set_xticks(range(0, 24, 3), [f"{order[i]:02d}:00" for i in range(0, 24, 3)])
ax.yaxis.set_major_formatter("{x:.0%}")
plt.show()

## Finding 5: delays cross borders

German ICEs run into the Netherlands, Belgium and Switzerland. The same trains, measured at
stations inside each country, next to that country's own intercity trains. In Switzerland, SBB
is reported to replace late German trains at the border, so look at the cancelled share too.
`SBB` in Germany = trains run by the Swiss railway around Basel and Lake Constance.

In [ ]:
xb = q(f'''select country, train_type, {MEASURES} from a
           where train_type in ('ICE', 'IC', 'Intercity', 'SBB') group by 1, 2''').set_index(["country", "train_type"])
pairs = [("DE", "ICE"), ("NL", "ICE"), ("NL", "Intercity"), ("BE", "ICE"), ("BE", "IC"),
         ("CH", "ICE"), ("CH", "IC"), ("DE", "SBB")]
xb.loc[pairs, ["arrivals", "under6", "cancelled"]]

## Finding 6: who cuts late trains short?

If a train is very late, a railway can end it early, turn it around and send it back on time.
The passengers further down the line wait for the next one, and a train that never arrived
can't be counted as late.

The test: at every station where a train was **meant to carry on**, how likely is it to be
terminated right there, depending on how late it arrived? (Normal termini are left out.)
This isn't proof that anyone games the figures: ending a late train early also keeps the
return trip on time.

In [ ]:
haz = q('''with r as (select *, seq = max(seq) over (partition by country, run_id) as is_last from a),
      s as (
        select country,
          case when delay_min <= 2 then '0-2' when delay_min <= 5 then '3-5'
               when delay_min <= 9 then '6-9' when delay_min <= 14 then '10-14'
               when delay_min <= 29 then '15-29' when delay_min <= 59 then '30-59' else '60+' end as bucket,
          min(delay_min) lo, count(*) arrivals_meant_to_continue, sum(cut_after::int) cut_here
        from r where not cancelled and (not is_last or cut_after) group by 1, 2)
      select *, cut_here / arrivals_meant_to_continue as p_cut_here from s order by country, lo''')
cut = haz.pivot(index="bucket", columns="country", values="p_cut_here").loc[
    ["0-2", "3-5", "6-9", "10-14", "15-29", "30-59", "60+"], ORDER]

ax = cut.rename(columns=NAMES).plot(color=[BLUE, ORANGE, "#7FA3D6", "#2E4F80"], lw=2.5, marker="o",
                                    title="Chance a train is cut short at a station, by how late it arrived")
ax.yaxis.set_major_formatter("{x:.0%}")
ax.set_xlabel("minutes late on arrival")
plt.show()
print("Compared with an on-time (0–2 min) train, at 30–59 min late:")
print((cut.loc["30-59"] / cut.loc["0-2"]).rename(NAMES).round(0).to_string())

runs = q('''with r as (select country, category, run_id, delay_min, cut_after from a where not cancelled)
            select country, count(distinct run_id) runs, sum(cut_after::int) / count(distinct run_id) share_runs_cut,
                   median(delay_min) filter (where cut_after) median_delay_at_cut
            from r where category = 'long' group by 1''').set_index("country")
print("\nLong-distance runs cut short, and how late the typical one was when cut:")
runs.loc[ORDER]

## Finding 7: cancellations

For Germany, the Netherlands and Switzerland: the share of train runs with at least one
cancelled stop. Belgium's raw data has no fully cancelled trains, so for Belgium this uses
Infrabel's own monthly count of trains fully or partly cancelled. That series currently stops in
February 2026, so it only covers the winter months.

In [ ]:
rc = q('''with r as (select country, run_id, bool_or(cancelled) any_c from a where country <> 'BE' group by 1, 2)
          select country, avg(any_c::int) share from r group by 1''').set_index("country").share
be_c = pd.read_csv(DATA / "be" / "official-cancellations.csv", sep=";", encoding="utf-8-sig")
be_c = be_c[be_c.maand.between(MONTHS[0], MONTHS[-1])]
rc["BE"] = be_c.sup.sum() / be_c.aantal_treinen.sum()
print(f"Belgium (Infrabel's count) covers {be_c.maand.min()} to {be_c.maand.max()}")
hbar(rc.loc[["NL", "DE", "CH", "BE"]].rename(NAMES), "Trains with at least one cancelled stop",
     fmt="{:.1%}", hot="Netherlands", xmax=0.12)

day = q(f"select country, day, {MEASURES} from a where country = 'NL' group by 1, 2 order by 2")
ax = day.plot(x="day", y="cancelled", color=ORANGE, legend=False, lw=1.5,
              title="Share of Dutch train stops cancelled, every day")
ax.yaxis.set_major_formatter("{x:.0%}")
plt.show()
print("Worst days (5–7 January: snow, frozen switches, an IT failure; 9 September: national strike):")
display(day.nlargest(5, "cancelled")[["day", "cancelled"]])

print("Under 6 min, with a cancelled stop counted as a fail:")
allc.loc[["DE", "NL", "CH"], ["under6", "under6_incl_cancel"]].rename(NAMES)

## What it means for you: a trip with one change

The chance that at least one of two long-distance trains is 15+ minutes late. A simple
estimate: it treats the two trains as independent.

In [ ]:
late15 = cat.xs("long", level="category").late15
two = 1 - (1 - late15) ** 2
hbar(two.loc[["CH", "NL", "BE", "DE"]].rename(NAMES),
     "Two long-distance trains: chance at least one is 15+ min late", fmt="{:.1%}", hot="Germany")

## Summary

| | Germany | Belgium | Netherlands | Switzerland |
|---|---|---|---|---|
| All trains under 6 min | 84% | 94% | 96% | 99% |
| Long-distance under 6 min | 53% | 92% | 94% | 97% |
| Long-distance 15+ min late | 27% | 3% | 2% | 1% |
| Trains with a cancelled stop | 6.8% | 3.8%* | 8.6% | 4.7% |
| Two-train trip, one 15+ min late | 47% | 5.3% | 3.5% | 1.6% |

\* Infrabel's own count, November 2025 – February 2026.

Of these four, Germany is last, and it's mostly the long-distance trains: German regional trains
are only a few points behind the others. Every railway's own headline number looks better than
this one ruler: neutralised figures, trains ended early, cancelled trains left out.

**Limits:** four countries, not all of Europe. Some German and Swiss times are the last
forecast shown to passengers rather than a measured time. Belgium's raw data has no fully
cancelled trains. The two-train odds treat the trains as independent.